# Baseline B — ACIT (JAAD)
**ACIT: Attention-Guided Cross-Modal Interaction Transformer** (Li & Müller, arXiv:2511.20020), reimplemented in
PyTorch from the paper (no official code). Inputs: local RGB crop, local optical flow, global semantic map, global
optical flow (all encoded by a frozen Swin-V2-B, 8×8×1024), ego-vehicle action as speed, pedestrian box.
Training as in the paper: BCE with class weights, Adam lr 2e-5, dropout 0.3, L2 1e-3 on the MLP.
Runs on both T4s with DDP (`torchrun`), fp16 AMP.

**Kaggle settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** ·
**Add Input → Datasets → `christopheronyiuke/jaad-video`** (346 official JAAD mp4 clips, mounted at
`/kaggle/input/jaad-video/JAAD_clips`). If the dataset is not attached, the notebook falls back to downloading the
official 3.1 GB zip from York University.

## Config

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'   # <-- set to your repo

WORK = '/kaggle/working'
REPO, JAAD, SAMPLES = f'{WORK}/repo', f'{WORK}/JAAD', f'{WORK}/samples'
CLIPS = '/kaggle/input/jaad-video/JAAD_clips'   # attached dataset christopheronyiuke/jaad-video
SUBSETS = ['beh']                 # ['beh', 'all'] -> JAAD_all needs ~30 GB of features in /tmp
SEEDS = [0, 1, 2]
EPOCHS = 40                       # paper doesn't state epochs/batch size; early stopping on val AUC (patience 10)
BATCH_SIZE = 16                   # global batch (8 per GPU)

FEATS = '/tmp/jaad_feats'         # use f'{WORK}/jaad_feats' (beh only, ~9.4 GB) to save it with the notebook output
FEATS_FROM_DATASET = None         # or a path under /kaggle/input/... with features saved by an earlier run
NEED_CLIPS = not FEATS_FROM_DATASET

## 1. Code

In [ ]:
!git clone -q {REPO_URL} {REPO} 2>/dev/null || git -C {REPO} pull -q
%cd {REPO}
!git log -1 --format='code commit: %H'
!pip install -q -U timm
!nvidia-smi -L
!df -h /kaggle/working /tmp

## 2. Dataset

In [ ]:
# Official JAAD annotations + split_ids + jaad_data.py (GitHub, ~50 MB). Clips: symlinked from the attached Kaggle
# dataset, or downloaded from York if it is not attached.
import os
if not NEED_CLIPS:
    clips_arg = ''
elif os.path.isdir(CLIPS):
    clips_arg = f'--clips_from {CLIPS}'
else:
    print('Kaggle dataset not attached -> downloading the official clips zip (~3.1 GB)')
    clips_arg = '--clips'
!python download_jaad.py --root {JAAD} {clips_arg}

## Samples and split
PCPA benchmark protocol (also used by ACIT): official video split (177 train / 29 val / 117 test videos),
16 observed frames ending 30-60 frames (1-2 s) before the crossing event, windows with 80 % overlap (stride 3),
tracks shorter than 76 frames dropped. Label = whether the pedestrian crosses. JAAD_beh = behaviour-annotated
pedestrians; JAAD_all adds the non-behavioural ones (label 0).

In [ ]:
for sub in SUBSETS:
    !python build_jaad_samples.py --jaad_root {JAAD} --out {SAMPLES} --sample_type {sub}

import pandas as pd
for sub in SUBSETS:
    man = pd.read_csv(f'{SAMPLES}/jaad_{sub}_manifest.csv')
    print(f'JAAD_{sub}'); display(man.groupby('split').agg(videos=('vid', 'nunique'), peds=('pid', 'nunique'),
                                                           samples=('label', 'size'), positives=('label', 'sum')))

## 3. Visual preprocessing (the expensive part, ~1 h for JAAD_beh on 2x T4)

In [ ]:
# RAFT-large flow + SegFormer-B5 (Cityscapes) + frozen Swin-V2-B -> [n, 64, 1024] fp16 per frame / crop, cached per video.
# Videos are sharded over the two T4s. Resumable: re-running skips videos that are already done.
import shutil
if FEATS_FROM_DATASET:
    FEATS = FEATS_FROM_DATASET
    print('using precomputed features from', FEATS)
else:
    src = f'{SAMPLES}/jaad_all.pkl' if 'all' in SUBSETS else f'{SAMPLES}/jaad_beh.pkl'
    need_gb = 30 if 'all' in SUBSETS else 9.4
    free_gb = shutil.disk_usage(os.path.dirname(FEATS.rstrip('/')) or '/').free / 1e9
    print(f'feature cache needs ~{need_gb} GB, free at {FEATS}: {free_gb:.0f} GB')
    !torchrun --nproc_per_node=2 extract_jaad_visual.py --jaad_root {JAAD} --samples {src} --out {FEATS} --vis 2

In [ ]:
# What Swin sees: local RGB | local flow | global RGB | global semantic map | global flow
import glob
from IPython.display import Image
for p in sorted(glob.glob(f'{FEATS}/vis/*.jpg'))[:6]:
    print(os.path.basename(p)); display(Image(p))

## 4. Sanity check: ACIT shapes, gradients, parameter count (paper: 5.15 M)

In [ ]:
import torch
from models.acit import ACIT
m = ACIT()
b = {k: torch.randn(2, 16, 64, 1024) for k in ['local_rgb', 'local_flow', 'global_seg', 'global_flow']}
b.update(speed=torch.randint(0, 5, (2, 16, 1)).float(), box=torch.rand(2, 16, 4))
(m(b).sum() + m.reg_loss()).backward()
print(f'{sum(p.numel() for p in m.parameters()) / 1e6:.2f} M params; params without grad:',
      [n for n, p in m.named_parameters() if p.grad is None])

## 5. Train + evaluate ACIT (DDP on both T4s)

In [ ]:
os.makedirs('logs', exist_ok=True)
os.environ['NCCL_P2P_DISABLE'] = '1'   # avoids NCCL peer-to-peer hangs seen on some T4 x2 hosts
for sub in SUBSETS:
    for seed in SEEDS:
        run = f'acit_{sub}_s{seed}'
        if os.path.exists(f'runs/{run}/metrics.json'):
            continue
        cmd = (f'torchrun --nproc_per_node=2 train.py --model acit --samples {SAMPLES}/jaad_{sub}.pkl --feats {FEATS} '
               f'--epochs {EPOCHS} --batch_size {BATCH_SIZE} --seed {seed} --out runs/{run}')
        !{cmd} 2>&1 | tee logs/{run}.log | grep -E "^(acit|ep|TEST)"

## Results
`bench_*`: PCPA-benchmark / paper style at threshold 0.5, where **`bench_auc` is ROC-AUC of the rounded predictions**
(that's the number published tables report). `auc`: real ROC-AUC. `thr_*`: threshold chosen on validation.

Paper (Table I) for reference — JAAD_beh: acc .70, AUC .62, F1 .79, P .69, R .93 · JAAD_all: .89, .83, .69, .65, .74.

In [ ]:
import glob
from evaluate import results_table
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
table = results_table(glob.glob('runs/acit_*'))
table.to_csv('runs/baseline_b_results.csv', index=False)
display(table)

In [ ]:
!zip -qr /kaggle/working/results.zip runs logs -x "*.pt"
!ls -lh /kaggle/working/results.zip